# SETU-RAG — Colab T4 walkthrough

Code-Switch-Aware Multilingual RAG for 22 Indian languages (customer support).
Run top-to-bottom on a **T4** runtime.

## 1 · Install
IndicLID and the IndicTrans2 toolkit are installed from GitHub (not on PyPI).

In [ ]:
!pip -q install -r requirements.txt
# IndicLID + IndicTrans2 toolkit:
# !git clone https://github.com/AI4Bharat/IndicLID.git
# !git clone https://github.com/AI4Bharat/IndicTrans2.git && pip -q install ./IndicTrans2/huggingface_interface/IndicTransToolkit

## 2 · Front-end: LID + CMI on a Hinglish query (no GPU needed)

In [ ]:
from setu_rag.front_end.language_id import LanguageIdentifier
from setu_rag.front_end.cmi import compute_cmi
lid = LanguageIdentifier().load()
q = 'mera refund kab tak aayega, order cancel kar diya tha'
toks = lid.tag(q)
print(compute_cmi(toks))

## 3 · Adaptive routing by CMI  [novel #1]

In [ ]:
from setu_rag.router.adaptive_router import decide_route
cmi = compute_cmi(toks)
frac_roman = sum(t.script=='Latn' for t in toks)/len(toks)
print(decide_route(cmi, frac_roman))

## 4 · Build the index (BGE-M3) and wire the pipeline

In [ ]:
# from setu_rag.retrieval.embedder import M3Embedder
# from setu_rag.retrieval.index import HybridIndex
# from setu_rag.pipeline import SetuRAG
# embedder = M3Embedder().load(); index = HybridIndex(embedder, SETTINGS)
# ... add data/faqs.sample.jsonl forms to the index ...
# rag = SetuRAG(index=index, translator=indictrans2)
# print(rag.answer(q).answer)

## 5 · CMI-conditioned prompt preview  [novel #3]

In [ ]:
from setu_rag.generation.generator import build_prompt
ctx = [{'answer':'Refund 5-7 business days mein aa jata hai.'}]
print(build_prompt(q, ctx, cmi.matrix_lang, cmi.cmi))

## 6 · CS-RAGAS metrics

In [ ]:
from setu_rag.eval import cs_metrics
print('CMI-alignment:', cs_metrics.cmi_alignment(q, 'aapka refund 5-7 din mein aa jayega', lid))